In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_sessions.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_events.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-7_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-5_task-BreathCounting_electrodes.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-10_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-4_task-BreathCounting_eeg.json
/kaggle

In [2]:
"""Kaggle-ready Extra Trees comparison: raw EEG versus ICA-cleaned EEG.

The script creates out-of-fold classification metrics and confusion matrices,
then fits a descriptive full-data Extra Trees model per condition to summarize
feature importance by frequency band.  "ICA-cleaned" denotes ICA-based
artifact cleaning.
"""

import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import gc
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import trapezoid
from scipy.signal import butter, hilbert, sosfiltfilt
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline


# Kaggle paths and analysis configuration

DATASET_ROOT = Path("/kaggle/input/datasets/jvkrishwanth/mwdataset")
OUTDIR = Path("/kaggle/working/dataset2_extratrees_raw_vs_ica_cleaned")
PLOT_DIR = OUTDIR / "plots"

SUBJECTS = ["sub-01", "sub-02"]
SESSIONS = range(1, 12)
TARGET_SFREQ = 256.0

# The selected BDF files are distributed across these Kaggle archive folders.
BDF_ARCHIVES = {
    "sub-01": {
        1: "sub-01-20260605T131512Z-3-002", 2: "sub-01-20260605T131512Z-3-003",
        3: "sub-01-20260605T131512Z-3-002", 4: "sub-01-20260605T131512Z-3-002",
        5: "sub-01-20260605T131512Z-3-002", 6: "sub-01-20260605T131512Z-3-001",
        7: "sub-01-20260605T131512Z-3-001", 8: "sub-01-20260605T131512Z-3-001",
        9: "sub-01-20260605T131512Z-3-001", 10: "sub-01-20260605T131512Z-3-003",
        11: "sub-01-20260605T131512Z-3-003",
    },
    "sub-02": {
        1: "sub-02-20260605T131514Z-3-003", 2: "sub-02-20260605T131514Z-3-002",
        3: "sub-02-20260605T131514Z-3-002", 4: "sub-02-20260605T131514Z-3-001",
        5: "sub-02-20260605T131514Z-3-001", 6: "sub-02-20260605T131514Z-3-002",
        7: "sub-02-20260605T131514Z-3-002", 8: "sub-02-20260605T131514Z-3-001",
        9: "sub-02-20260605T131514Z-3-001", 10: "sub-02-20260605T131514Z-3-001",
        11: "sub-02-20260605T131514Z-3-003",
    },
}
METADATA_ARCHIVES = {
    "sub-01": "sub-01-20260605T131512Z-3-001",
    "sub-02": "sub-02-20260605T131514Z-3-001",
}

# BDF Status event codes and Focus (1 to 6 s) / MW (-5 to -0.1 s) epoch definitions.
EVENT_TRIAL_START = 10
EVENT_MW_REPORT = 30
EVENT_START_COUNTING = 50
EPOCH_DURATION = 5.0
MW_START_OFFSET = -5.0       # MW epoch spans -5 to -0.1 s before a report.
FOCUS_START_OFFSET = 1.0      # Focus epoch spans +1 to +6.0 s after an anchor.

# Sliding windows turn each Focus (1 to 6 s) and MW (-5 to -0.1 s) epoch into repeated feature samples.
WINDOW_LENGTH = 1.5
WINDOW_STEP = 0.5
BANDS = {
    "Delta": (1.0, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 12.0),
    "Beta": (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
BAND_ORDER = ["Theta", "Alpha", "Gamma", "Beta", "Delta"]

CONDITIONS = ["Raw", "ICA-cleaned"]
CONDITION_COLORS = {"Raw": "#66c2a5", "ICA-cleaned": "#fc8d62"}
METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]


# Plot style: deliberately matched across all requested figures

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "axes.grid": True,
    "grid.color": "#c7c7c7",
    "grid.linewidth": 1.0,
    "axes.edgecolor": "#c7c7c7",
    "axes.linewidth": 1.0,
})


# Dataset loading, preprocessing, and ICA cleaning

def session_paths(subject, session):
    """Return deterministic BDF and channel-table paths for one recording."""
    session_label = f"ses-{session}"
    eeg_dir = DATASET_ROOT / BDF_ARCHIVES[subject][session] / subject / "eeg"
    metadata_dir = DATASET_ROOT / METADATA_ARCHIVES[subject] / subject / "eeg"
    return (
        eeg_dir / f"{subject}_{session_label}_task-BreathCounting_eeg.bdf",
        metadata_dir / f"{subject}_{session_label}_task-BreathCounting_channels.tsv",
    )


def deduplicate_events(events):
    """Remove duplicate sample/code event pairs after event-index resampling."""
    if len(events) == 0:
        return events
    event_df = pd.DataFrame(events, columns=["sample", "previous", "code"])
    event_df = event_df.drop_duplicates(["sample", "code"], keep="first")
    return event_df.sort_values(["sample", "code"])[["sample", "previous", "code"]].to_numpy(dtype=int)


def channel_names(raw, channels_tsv):
    """Identify recording-present EEG channels and the available EXG reference channels."""
    channel_table = pd.read_csv(channels_tsv, sep="\t")
    eeg_names = channel_table.loc[
        channel_table["channelTypes"].fillna("").str.upper().eq("EEG"), "name"
    ].astype(str).tolist()
    eeg_names = [channel for channel in eeg_names if channel in raw.ch_names]
    exg_names = [f"EXG{number}" for number in range(1, 9) if f"EXG{number}" in raw.ch_names]
    if not eeg_names:
        raise RuntimeError(f"No EEG channels from {channels_tsv.name} are present in the BDF file.")
    return eeg_names, exg_names


def ica_clean(raw, eeg_names, exg_names, correlation_threshold=0.30):
    """Remove ICA components correlated with EXG channels without dropping trials."""
    cleaned = raw.copy()
    if not exg_names:
        return cleaned.pick(eeg_names)
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica", random_state=42, max_iter="auto",
    )
    ica.fit(cleaned, picks=eeg_names, verbose=False)
    scores = [
        np.asarray(ica.score_sources(cleaned, target=cleaned.get_data(picks=[name])[0]), dtype=float)
        for name in exg_names
    ]
    excluded = np.flatnonzero(np.max(np.abs(np.vstack(scores)), axis=0) >= correlation_threshold)
    ica.apply(cleaned, exclude=excluded, verbose=False)
    return cleaned.pick(eeg_names)

def build_epoch_events(events, sfreq):
    """Create Focus=0 and MW=1 event starts for the configured Focus (1 to 6 s) and MW (-5 to -0.1 s) epochs."""
    rows = []
    for sample, _, code in events:
        if code == EVENT_MW_REPORT:
            start = sample + round(MW_START_OFFSET * sfreq)
            label = 1
        elif code in (EVENT_TRIAL_START, EVENT_START_COUNTING):
            start = sample + round(FOCUS_START_OFFSET * sfreq)
            label = 0
        else:
            continue
        if start >= 0:
            rows.append([start, 0, label])
    return deduplicate_events(np.asarray(rows, dtype=int)) if rows else np.empty((0, 3), dtype=int)


def load_conditions(subject, session):
    """Load one BDF and return matched raw and ICA-cleaned EEG epoch sources."""
    bdf_path, channels_tsv = session_paths(subject, session)
    if not bdf_path.is_file() or not channels_tsv.is_file():
        raise FileNotFoundError(f"Missing input for {subject} session {session}: {bdf_path} / {channels_tsv}")

    raw = mne.io.read_raw_bdf(bdf_path, preload=True, stim_channel="Status", verbose=False)
    events = mne.find_events(raw, stim_channel="Status", shortest_event=1, consecutive=True, verbose=False)
    eeg_names, exg_names = channel_names(raw, channels_tsv)
    original_sfreq = raw.info["sfreq"]

    raw.pick(eeg_names + exg_names)
    raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
    events = events.copy()
    events[:, 0] = np.rint(events[:, 0] * TARGET_SFREQ / original_sfreq).astype(int)
    events = deduplicate_events(events)
    raw.filter(0.5, 45.0, fir_design="firwin", verbose=False)

    # Raw condition: filtered EEG only.  No EXG contribution is removed.
    raw_condition = raw.copy().pick(eeg_names)
    # Clean condition: the same filtered data after ICA cleaning.
    clean_condition = ica_clean(raw, eeg_names, exg_names)
    return {"Raw": raw_condition, "ICA-cleaned": clean_condition}, events


def make_epochs(raw, events):
    epoch_events = build_epoch_events(events, raw.info["sfreq"])
    if not len(epoch_events):
        raise RuntimeError("No valid MW or Focus events are available for epoching.")
    
    sfreq = raw.info["sfreq"]
    focus_samples = round(5.0 * sfreq)
    mw_samples = round(4.9 * sfreq)
    pad_samples = focus_samples - mw_samples
    
    epochs_data = []
    kept_events = []
    for event in epoch_events:
        start = event[0]
        label = event[2]
        stop = start + mw_samples if label == 1 else start + focus_samples
        if start >= 0 and stop <= raw.n_times:
            data = raw.get_data(start=start, stop=stop)
            if label == 1:
                data = np.pad(data, ((0,0), (0, pad_samples)), mode='edge')
            epochs_data.append(data)
            kept_events.append(event)
            
    if not epochs_data:
        raise RuntimeError("No valid MW or Focus epochs are within bounds.")
        
    return mne.EpochsArray(
        np.asarray(epochs_data), raw.info.copy(), tmin=0.0,
        events=np.asarray(kept_events, dtype=int),
        event_id={"Focus": 0, "MW": 1}, verbose=False,
    )


# Windowed, band-wise feature extraction

def bandpass_filter(data, low, high, sfreq):
    nyquist = sfreq / 2.0
    sos = butter(4, [low / nyquist, high / nyquist], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def burst_count(envelope):
    """Count upward crossings of mean + 2 SD in one amplitude envelope."""
    threshold = envelope.mean() + 2.0 * envelope.std()
    return float(np.sum(np.diff((envelope > threshold).astype(int)) == 1))


def window_features(epoch_data, sfreq, state=None):
    """Return one per-channel feature dictionary for every overlapping window."""
    window_samples = round(WINDOW_LENGTH * sfreq)
    step_samples = round(WINDOW_STEP * sfreq)
    features = []

    for start in range(0, epoch_data.shape[-1] - window_samples + 1, step_samples):
        if state == "MW" and start + window_samples > round(4.9 * sfreq):
            continue
        data = epoch_data[:, start:start + window_samples]
        values = {}
        for band, (fmin, fmax) in BANDS.items():
            filtered = bandpass_filter(data, fmin, fmax, sfreq)
            psd = np.abs(np.fft.rfft(filtered, axis=-1)) ** 2
            freqs = np.fft.rfftfreq(window_samples, d=1.0 / sfreq)
            mask = (freqs >= fmin) & (freqs <= fmax)
            total_power = trapezoid(psd, freqs, axis=-1)
            band_power = trapezoid(psd[:, mask], freqs[mask], axis=-1)
            values[f"{band}_abs"] = band_power
            values[f"{band}_rel"] = band_power / (total_power + 1e-12)

            if band in ("Theta", "Alpha"):
                envelope = np.abs(hilbert(filtered, axis=-1))
                mean_envelope = envelope.mean(axis=-1)
                values[f"{band}_mean_env"] = mean_envelope
                values[f"{band}_var_env"] = envelope.var(axis=-1)
                values[f"{band}_cv_env"] = envelope.std(axis=-1) / (mean_envelope + 1e-12)
                values[f"{band}_bursts"] = np.asarray([burst_count(signal) for signal in envelope])

            if band == "Alpha":
                alpha_envelope = np.abs(hilbert(filtered, axis=-1))
                correlation = np.corrcoef(alpha_envelope)
                np.fill_diagonal(correlation, np.nan)
                values["Alpha_sync"] = np.nanmean(correlation, axis=0)
        features.append(values)
    return features


def extract_features(epochs, subject, session, condition):
    """Create a long feature table with one row per epoch-window-channel."""
    rows = []
    for epoch_index, (epoch_data, event) in enumerate(zip(epochs.get_data(), epochs.events), start=1):
        state = "MW" if event[2] == 1 else "Focus"
        for window_index, features in enumerate(window_features(epoch_data, epochs.info["sfreq"], state=state)):
            for channel_index, channel in enumerate(epochs.ch_names):
                row = {
                    "Condition": condition,
                    "Subject": subject,
                    "Session": session,
                    "Subject_Session": f"{subject}_ses-{session}",
                    "Epoch": epoch_index,
                    "Window": window_index,
                    "State": state,
                    "Label": int(state == "MW"),
                    "Channel": channel,
                }
                for feature, values in features.items():
                    row[feature] = values[channel_index]
                rows.append(row)
    return pd.DataFrame(rows)


def make_wide_table(long_features):
    """Convert channel rows into classifier rows while preserving group labels."""
    metadata = ["Condition", "Subject", "Session", "Subject_Session", "Epoch", "Window", "State", "Label"]
    feature_columns = [column for column in long_features.columns if column not in metadata + ["Channel"]]
    wide = long_features.pivot_table(
        index=metadata,
        columns="Channel",
        values=feature_columns,
        aggfunc="first",
    ).reset_index()
    wide.columns = [
        f"{feature}__{channel}" if channel else feature
        for feature, channel in wide.columns.to_flat_index()
    ]
    return wide


# Extra Trees evaluation and importance summaries

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=500,
            max_depth=20,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])


def make_cv(labels, groups):
    n_splits = min(5, len(np.unique(groups)))
    if n_splits < 2:
        raise ValueError("At least two independent groups are required for grouped cross-validation.")
    try:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    except ImportError:
        return GroupKFold(n_splits=n_splits)


def evaluate_condition(wide_table):
    """Return out-of-fold labels/probabilities and five classification metrics."""
    metadata = {"Condition", "Subject", "Session", "Subject_Session", "Epoch", "Window", "State", "Label"}
    feature_columns = [column for column in wide_table.columns if column not in metadata]
    X = wide_table[feature_columns]
    y = wide_table["Label"].to_numpy()
    groups = wide_table["Subject_Session"].to_numpy()
    cv = make_cv(y, groups)
    oof_prediction = np.full(len(y), -1, dtype=int)
    oof_probability = np.full(len(y), np.nan, dtype=float)

    for train_index, test_index in cv.split(X, y, groups):
        if len(np.unique(y[train_index])) < 2:
            continue
        model = make_model()
        model.fit(X.iloc[train_index], y[train_index])
        oof_prediction[test_index] = model.predict(X.iloc[test_index])
        oof_probability[test_index] = model.predict_proba(X.iloc[test_index])[:, 1]

    valid = oof_prediction >= 0
    if valid.sum() == 0:
        raise RuntimeError("No valid out-of-fold predictions were generated.")
    y_true = y[valid]
    y_pred = oof_prediction[valid]
    y_probability = oof_probability[valid]
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_probability),
    }
    return feature_columns, y_true, y_pred, y_probability, metrics


def band_importance(wide_table, feature_columns):
    """Fit on all data for descriptive band importance; not a performance estimate."""
    model = make_model()
    model.fit(wide_table[feature_columns], wide_table["Label"])
    importance = pd.DataFrame({
        "Feature": feature_columns,
        "Importance": model.named_steps["model"].feature_importances_,
    }).sort_values("Importance", ascending=False)
    totals = {}
    for band in BAND_ORDER:
        totals[band] = importance.loc[importance["Feature"].str.startswith(f"{band}_"), "Importance"].sum()
    total_imp = sum(totals.values())
    if total_imp > 0:
        for band in BAND_ORDER:
            totals[band] /= total_imp
    return importance, pd.DataFrame({"Band": BAND_ORDER, "Importance": [totals[band] for band in BAND_ORDER]})


# Publication-style plots

def save_band_plot(band_table, ymax):
    colors = sns.color_palette("viridis", n_colors=len(BAND_ORDER))
    figure, axis = plt.subplots(figsize=(16, 10))
    axis.bar(band_table["Band"], band_table["Importance"], color=colors, width=0.8)
    axis.set_title("Total Extra Trees Feature Importance by Band", pad=12)
    axis.set_xlabel("Band")
    axis.set_ylabel("Importance")
    axis.set_ylim(0, ymax)
    figure.tight_layout()
    figure.savefig(PLOT_DIR / "bandwise_feature_importance.png", bbox_inches="tight")
    plt.close(figure)


def save_confusion_plot(y_true, y_pred, condition, vmax):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        square=True,
        vmin=0,
        vmax=vmax,
        xticklabels=["Focused", "MW"],
        yticklabels=["Focused", "MW"],
        ax=axis,
    )
    axis.set_title(f"{condition}: out-of-fold confusion matrix", pad=12)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"{condition.lower().replace('-', '_')}_confusion_matrix.png", bbox_inches="tight")
    plt.close(figure)


def save_metric_plot(metric_rows):
    metric_table = pd.DataFrame(metric_rows)
    figure, axis = plt.subplots(figsize=(20, 12))
    positions = np.arange(len(METRIC_ORDER))
    width = 0.4
    for offset, condition in [(-width / 2, "Raw"), (width / 2, "ICA-cleaned")]:
        values = metric_table.loc[metric_table["Condition"].eq(condition), "Score"].to_numpy()
        bars = axis.bar(positions + offset, values, width=width, color=CONDITION_COLORS[condition], label=condition)
        axis.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=12)
    axis.set_title("Extra Trees: raw vs ICA-cleaned EEG classification", pad=12)
    axis.set_xlabel("Metric")
    axis.set_ylabel("Score")
    axis.set_xticks(positions, METRIC_ORDER)
    axis.set_ylim(0, 1.0)
    axis.legend(title="Condition", loc="upper right")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / "raw_vs_ica_cleaned_classification_metrics.png", bbox_inches="tight")
    plt.close(figure)


# Main execution

def main():
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f"Kaggle dataset directory not found: {DATASET_ROOT}")
    OUTDIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    long_tables = []
    failures = []
    for subject in SUBJECTS:
        for session in SESSIONS:
            try:
                print(f"Processing {subject}, session {session}")
                conditions, events = load_conditions(subject, session)
                for condition, raw in conditions.items():
                    epochs = make_epochs(raw, events)
                    long_tables.append(extract_features(epochs, subject, session, condition))
                    del epochs
                del conditions, events
                gc.collect()
            except Exception as error:
                print(f"FAILED {subject}, session {session}: {error}")
                failures.append({"Subject": subject, "Session": session, "Error": str(error)})

    if not long_tables:
        raise RuntimeError("No recordings were processed successfully.")
    long_features = pd.concat(long_tables, ignore_index=True)
    long_features.to_csv(OUTDIR / "windowed_features_raw_and_ica_cleaned.csv", index=False)
    pd.DataFrame(failures).to_csv(OUTDIR / "failed_runs.csv", index=False)

    results = {}
    metric_rows = []
    for condition in CONDITIONS:
        wide = make_wide_table(long_features[long_features["Condition"].eq(condition)].copy())
        feature_columns, y_true, y_pred, y_probability, metrics = evaluate_condition(wide)
        importance, bands = band_importance(wide, feature_columns)
        wide.to_csv(OUTDIR / f"{condition.lower().replace('-', '_')}_wide_features.csv", index=False)
        importance.to_csv(OUTDIR / f"{condition.lower().replace('-', '_')}_feature_importance.csv", index=False)
        bands.to_csv(OUTDIR / f"{condition.lower().replace('-', '_')}_bandwise_feature_importance.csv", index=False)
        results[condition] = {"y_true": y_true, "y_pred": y_pred, "bands": bands}
        metric_rows.extend({"Condition": condition, "Metric": metric, "Score": score} for metric, score in metrics.items())

    metrics_df = pd.DataFrame(metric_rows)
    metrics_df["Metric"] = pd.Categorical(metrics_df["Metric"], METRIC_ORDER, ordered=True)
    metrics_df = metrics_df.sort_values(["Metric", "Condition"])
    metrics_df.to_csv(OUTDIR / "raw_vs_ica_cleaned_classification_metrics.csv", index=False)

    common_band_ymax = max(result["bands"]["Importance"].max() for result in results.values()) * 1.08
    common_confusion_vmax = max(confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1]).max() for result in results.values())
    save_band_plot(results["ICA-cleaned"]["bands"], common_band_ymax)
    for condition in CONDITIONS:
        save_confusion_plot(results[condition]["y_true"], results[condition]["y_pred"], condition, common_confusion_vmax)
    save_metric_plot(metric_rows)

    print("\nOut-of-fold Extra Trees metrics:")
    print(metrics_df.pivot(index="Metric", columns="Condition", values="Score"))
    print(f"\nSaved CSV files to: {OUTDIR}")
    print(f"Saved plots to: {PLOT_DIR}")


if __name__ == "__main__":
    main()


Processing sub-01, session 1
Processing sub-01, session 2
Processing sub-01, session 3
Processing sub-01, session 4
Processing sub-01, session 5
Processing sub-01, session 6
Processing sub-01, session 7
Processing sub-01, session 8
Processing sub-01, session 9
Processing sub-01, session 10
Processing sub-01, session 11
Processing sub-02, session 1
Processing sub-02, session 2
Processing sub-02, session 3
Processing sub-02, session 4
Processing sub-02, session 5
Processing sub-02, session 6
Processing sub-02, session 7
Processing sub-02, session 8
Processing sub-02, session 9
Processing sub-02, session 10
Processing sub-02, session 11

Out-of-fold Extra Trees metrics:
Condition  ICA-cleaned       Raw
Metric                          
Accuracy      0.695471  0.707743
Precision     0.728244  0.737345
Recall        0.433111  0.467312
F1            0.543177  0.572064
ROC-AUC       0.748918  0.764974

Saved CSV files to: /kaggle/working/dataset2_extratrees_raw_vs_ica_cleaned
Saved plots to: /